# Non-pivot xslx functions 1

Created: 2026-09-18

Develop functions for accessing data in xlsx workbooks that do not contain pivot tables.

Pivot workbooks constitute only a small fraction of the Excel files provided by e-Stat. Other xslx files may require ad hoc development. The purpose of this notebook is use Python package [`openpyxl`](https://pypi.org/project/openpyxl/) ([documentation: ](https://openpyxl.readthedocs.io/en/stable/)) to explore a handfull of non-pivot workbooks looking for automation hints. `Openpyxl` only works for Excel 2010 and later. Xlsx files are zip files that contain an xml file.

## Install the necessary packages

In [1]:
%%capture capt
# %%capture puts the output in 'capt' from which it can be ignored or printed
%pip install pandas
%pip install fsspec # to help pandas IO. See https://pandas.pydata.org/docs/user_guide/io.html#reading-writing-remote-files
%pip install openpyxl

## Install the newest version of estatjp

Repeat during development to import revisions.

## Data source urls

Workbooks of immediate interest are in the [statistics on registered foreign residents that are provided by the Immigration Services Agency](https://www.moj.go.jp/isa/policies/statistics/toukei_ichiran_touroku.html). 

In [2]:
# 14-12-07 市区町村別　国籍・地域別　在留外国人 2014年12月
url141207 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000029226622&fileKind=0" # 14-12-07 - Dec 2014 by municipality (not pivot)
url2506t1 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000040379765&fileKind=0" # 25-06-t1 - June 2025 by prefecture (pivot)
url2506t2 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000040379766&fileKind=0" # 25-06-t2 - June 2025 by municipality (deleted pivot)

## Import the module and download a file

In [3]:
from estatjp import excel
from pathlib import Path

ROOT_DIR = Path.cwd().parent.parent
dataraw = ROOT_DIR / Path(".data/raw/isa")
newdata = excel.download_to_folder(url141207, dataraw, overwrite=True, verbose=True)
newdata

{'Content-Type': 'application/octet-stream', 'Transfer-Encoding': 'chunked', 'Connection': 'keep-alive', 'X-Cache-Status': 'NOTCACHED', 'Content-Encoding': 'gzip', 'X-Frame-Options': 'SAMEORIGIN', 'Cache-Control': 'private', 'X-XSS-Protection': '1; mode=block', 'Date': 'Wed, 30 Sep 2026 01:28:50 GMT', 'Accept-Ranges': 'bytes', 'Server': 'ZENEDGE', 'X-Content-Type-Options': 'nosniff', 'X-Zen-Fury': 'daef0452b7cebfe96f4ed82ea72ad774badce045', 'Content-Disposition': "attachment; filename*=UTF-8''14-12-07%20.xlsx", 'X-Cache': 'MISS', 'Strict-Transport-Security': 'max-age=0', 'X-Cdn': 'Served-By-Zenedge'}


WindowsPath('c:/Users/Alan/Documents/GitHub/estatjp/.data/raw/isa/14-12-07 .xlsx')

## Examine the worksheets and look for pivot tables

In [4]:
import openpyxl
wb = openpyxl.load_workbook(newdata, data_only=True)
snames = wb.sheetnames
print(snames)
has_pivot = False
has_pivot_cache_records = False
for ws in wb.worksheets:
    print('title: ' + ws.title)
    print('dimensions: ' + ws.dimensions)
    if ws.print_title_rows != None:
        print('print_title_rows: ' + ws.print_title_rows)
    print('_pivots len: ' + str(len(ws._pivots)))
    if len(ws._pivots) > 0:
        has_pivot = True
        print('pivot_table.cache.records: ' + str(ws._pivots[0].cache.records))
        if ws._pivots[0].cache.records != None:
            has_pivot_cache_records = True
    print('merged_cells len: ' + str(len(ws.merged_cells.ranges)))


['14-12-07 ']
title: 14-12-07 
dimensions: A2:M2301
print_title_rows: $1:$4
_pivots len: 0
merged_cells len: 1180


## Parse title to get the names of the columns we will use in the column-oriented output.

This section of the document borrows terminology from the [APA Style Manual](https://apastyle.apa.org/style-grammar-guidelines/tables-figures/tables). 

The titles of tables from e-Stat contain the information we need for the column headings in our output. The non-pivot workbook `url141207` is from the stats on foreign residents in Japan as of [December 2014](https://www.e-stat.go.jp/SG1/estat/List.do?lid=000001133760), specifically table 14-12-08. The title of this table is "市区町村別　国籍・地域別　在留外国人," which translates to "by municipality", "by nationality and region", and "resident foreigners". The first term provides the label for the column of row labels. The second term provides the label covering the column names. And the third term becomes the observed variable. 

The headings of Excel tables in e-Stat can be decked, but the current table has a single row of column headings. 

![Table 14-12-07 upper left corner](images/Table14-12-07-upper-left-corner.jpg) 

![Table 14-12-07 lower left corner](images/Table14-12-07-lower-left-corner.jpg) 

On the other hand, the row labels in the three columns at left are not so trivial. The header spans columns A-C. The prefecture name in column A (Hokkaido) is to be the right part of all municipal names below it until Aomori starts in row 263. The lower left corner of the table shows a comment under the table. 

This first algorithm takes advantage of the merge cells in the stub header. The text of the stub header is in cell A4 and spans columns B and C by means of merge cells.

In [12]:
from IPython.display import HTML

html_table = """
<table border="1">
    <caption>Label columns in worksheet vs desired label in output</caption>
    <thead>
    <tr>
    <th width="100" rowspan="2" >Row</th>
        <th align="center" width="300" colspan="3">Columns in worksheet table</th>
        <th rowspan="2">:</th>
        <th rowspan="2" align="center" width="200" >Desired row label in output</th>
    </tr>
    <tr><td align="center">A</td><td align="center">B</td><td align="center">C</td></tr>
    <tr>
        <td width="100">4</td>
        <td align="center" width="300" colspan="3">市区町村</td>
        <td>:</td>
        <td align="center" width="200" >市区町村</td>
    </tr>
    </thead>
    <tbody>
    
    <tr>
        <td  width="100">5</td>
        <td width="100" align="left" >北海道</td>
        <td align="left"  colspan="2" width="200">札幌市</td>
        <td>:</td>
        <td align="right" width="200" >北海道札幌市</td>
    </tr>
    <tr>
        <td width="100" >6</td>
        <td width="100"  ></td>
        <td width="100"  ></td>
        <td width="100" align="left" >中央区</td>
        <td>:</td>
        <td align="right" width="200" >北海道札幌市中央区</td>
    </tr>
    <tr>
        <td width="100" >7</td>
        <td width="100"  ></td>
        <td width="100"  ></td>
        <td width="100" align="left" >北区</td>
        <td>:</td>
        <td align="right" width="200" >北海道札幌市北区</td>
    </tr>
    <tr>
        <td width="100" >8</td>
        <td width="100"  ></td>
        <td width="100"  ></td>
        <td width="100" align="left" >東区</td>
        <td>:</td>
        <td align="right" width="200" >北海道札幌市東区</td>
    </tr>
    <tr><td>.</td></tr>
    <tr>
        <td  width="100">16</td>
        <td width="100" align="left" ></td>
        <td align="left"  colspan="2" width="200">函館市</td>
        <td>:</td>
        <td align="right" width="200" >北海道函館市</td>
    </tr>
    <tr><td>.</td></tr>
    <tr>
        <td  width="100">50</td>
        <td width="100" align="left" ></td>
        <td align="left"  colspan="2" width="200">石狩郡</td>
        <td>:</td>
        <td align="right" width="200" >北海道石狩郡</td>
    </tr>
    <tr>
        <td width="100" >51</td>
        <td width="100"  ></td>
        <td width="100"  ></td>
        <td width="100" align="left" >当別町</td>
        <td>:</td>
        <td align="right" width="200" >北海道石狩郡当別町</td>
    </tr>
    <tr><td>.</td></tr>
    <tr>
        <td  width="100">2299</td>
        <td width="300" align="left" colspan="3">不詳・不定</td>
        <td>:</td>
        <td align="right" width="200" >不詳・不定</td>
    </tr>
    </tbody>
    <tfoot>
        <tr>
            <td>Note 1:</td><td colspan="4">There are no values for counties (郡) in this worksheet, but all cells contain zero (0).</td>
        </tr>
    </tfoot>

</table>
"""

display(HTML(html_table))


## Use the print_title_rows attribute to parse column header information

The current example contains a [`worksheet.print_title_rows`](https://openpyxl.readthedocs.io/en/stable/api/openpyxl.worksheet.print_settings.html#openpyxl.worksheet.print_settings.PrintTitles.title) attribute. Try to use it to parse the column header information, both for the table to be read and for the column-oriented table to be created for the dataframe.

Reference the class [openpyxl.worksheet.print_settings](https://openpyxl.readthedocs.io/en/stable/_modules/openpyxl/worksheet/print_settings.html#PrintTitles.from_string)

### Use a [regular expression](https://docs.python.org/3/library/re.html) to identify and parse the title

Use an actual example to develop the regular expression. The group names 'cat01' and 'cat02' match those used by e-Stat's API service (See [Accessing the MOJ/ISA data via API calls](moj-isa-via-api.ipynb)).

In [ ]:
import re
TITLE_REGEX = re.compile(r"第\d+表\s+(?P<cat01_label>\S+)別\s+(?P<cat02_label>\S+)別\s+(?P<tab_label>\w+)(（(その\w+\s+)?(?P<sub_tab_label>\S+)）)?")
title_str = "第７表　市区町村別　国籍・地域別　在留外国人" 
m= TITLE_REGEX.match(title_str)
print(m)
print(m.group(1,2,3,4))
print(m.group('cat01_label'))
print(m.group('cat02_label'))
print(m.group('tab_label'))
print(m.group('sub_tab_label'))

<re.Match object; span=(0, 22), match='第７表\u3000市区町村別\u3000国籍・地域別\u3000在留外国人'>
('市区町村', '国籍・地域', '在留外国人', None)
市区町村
国籍・地域
在留外国人
None


## Locate and parse title and column names in worksheet

The logical columns, particularly the index column, may span multiple spreadsheet columns. Define a column_header class to contain the parsed results. See the [Python tutorial](https://docs.python.org/3/tutorial/classes.html) and class definitions of [Cell](https://openpyxl.readthedocs.io/en/stable/api/openpyxl.cell.cell.html#openpyxl.cell.cell.Cell) and [MergedCell](https://openpyxl.readthedocs.io/en/stable/api/openpyxl.cell.cell.html#openpyxl.cell.cell.MergedCell).

In [ ]:
from dataclasses import dataclass
#from typing import Optional
from openpyxl.cell.cell import Cell
from openpyxl.cell.cell import MergedCell

@dataclass
class column_header:
    """Contains information needed to convert the rows below it into column-oriented format. 

    The column category col_cat is either 'cat01' or 'cat02'.

    Sometimes the data is subdivided across multiple tables, with the restriction contained in the title.

    """
    __slots__ = (
        'col_cat',
        'col_cat_label',
        'sub_tab_label',
        'tab_label',
        'cell_list',
        'merged_cell_list'
                )

    def __init__(self, cellinstance, col_cat, col_cat_label, tab_label, sub_tab_label = None):
        self.col_cat = col_cat
        self.col_cat_label = col_cat_label
        self.tab_label = tab_label
        self.sub_tab_label = sub_tab_label
        self.cell_list = []
        self.merged_cell_list = []
        if not isinstance(cellinstance, Cell):
            raise TypeError(f"cellinstance must be Cell, got {type(cellinstance).__name__}")
        self.cell_list.append((cellinstance.row,cellinstance.column))

    def add_merged_cell(self, cellinstance):
        if isinstance(cellinstance, MergedCell):
            self.merged_cell_list.append((cellinstance.row,cellinstance.column))
        else:
            raise TypeError(f"cellinstance must be MergedCell, got {type(cellinstance).__name__}")

    def serialize(self, worksheet):
        col_head = ""
        for c in self.cell_list:
            col_head = col_head + str(worksheet.cell(c[0], c[1]).value).replace('\u3000','')
        r = {
                'col_cat': self.col_cat,
                'col_cat_label': self.col_cat_label,
                'sub_tab_label': self.sub_tab_label,
                'tab_label': self.tab_label,
                'col_head': col_head
            }
        return r

    def header_length(self):
        return len(self.cell_list) + len(self.merged_cell_list)



Parse the print title rows to get the title and the column headers.

In [13]:
import re
import datetime as dt
from openpyxl.worksheet import cell_range

SHEETNAME_REGEX = re.compile(r"(?P<year>\d{2})-(?P<month>\d{2})-(?P<tableno>\d{1,2}(-[\da-zA-Z0-9_]+)?)")
TITLE_REGEX = re.compile(r"第\d+表\s+(?P<cat01_label>\S+)別\s+(?P<cat02_label>\S+)別\s+(?P<tab_label>\w+)(（(その\w+\s+)?(?P<sub_tab_label>\S+)）)?")
ws = wb.active # There is only one worksheet in the current workbook
print(ws.print_title_rows)
print(ws.print_titles)
print(ws._print_rows.min_row)
print(ws._print_rows.max_row)
title_result = {}
# the year, month and table name are in ws.title
mt = SHEETNAME_REGEX.match(ws.title)
if mt != None:
    yr = int("20" + mt.group('year'))
    month = int(mt.group('month'))
    sd = dt.date(yr, month, 1)
    statsdtstr = str(sd.year) + "年" + str(sd.month) + "月"
    tcstr = "20" + mt.group('year') + "00" + mt.group('month') + mt.group('month')
    title_result['time_code'] = int(tcstr)
    title_result['stats_date'] = statsdtstr
    print(title_result['time_code'])

start_row = ws._print_rows.min_row
end_row = ws._print_rows.max_row
rowiter = ws.iter_rows(min_row=start_row, max_row=end_row)
row = next(rowiter, None)
while row != None and len(title_result) <= 2:
    # Parse until we get the title
    for cell in row:
        if cell.value != None:
            m = TITLE_REGEX.match(cell.value)
            if m != None:
                title_result["cat01_label"] = m.group('cat01_label')
                title_result["cat02_label"] = m.group('cat02_label')
                title_result["tab_label"] = m.group('tab_label')
                title_result["sub_tab_label"] = m.group('sub_tab_label')
    row = next(rowiter)

print(title_result)

# Find and parse the column header row
left_col = ws.min_column
right_col = ws.max_column
col_headers = []
while row != None:
    ch = None
    for c in row:
        if c.value != None and c.column == left_col:
            ch = column_header( 
                    c,
                    col_cat = "cat01",
                    col_cat_label = title_result["cat01_label"],
                    tab_label = title_result["tab_label"],
                    restriction = title_result["sub_tab_label"]
                    )
        elif isinstance(c, MergedCell):
            if ch != None:
                ch.add_merged_cell(c)
        elif c.value != None:
            col_headers.append(ch)
            ch = None
            ch = column_header( 
                    c,
                    col_cat = "cat02",
                    col_cat_label = title_result["cat02_label"],
                    tab_label = title_result["tab_label"],
                    restriction = title_result["sub_tab_label"]
                    )
    if ch != None:
        col_headers.append(ch)
    row = next(rowiter, None)

for h in col_headers:
    print(h.serialize(ws))

$1:$4
'14-12-07 '!$1:$4
1
4
2014001212
{'time_code': 2014001212, 'stats_date': '2014年12月', 'cat01_label': '市区町村', 'cat02_label': '国籍・地域', 'tab_label': '在留外国人', 'sub_tab_label': None}
{'col_cat': 'cat01', 'col_cat_label': '市区町村', 'restriction': None, 'tab_label': '在留外国人', 'col_head': '市区町村'}
{'col_cat': 'cat02', 'col_cat_label': '国籍・地域', 'restriction': None, 'tab_label': '在留外国人', 'col_head': '総数'}
{'col_cat': 'cat02', 'col_cat_label': '国籍・地域', 'restriction': None, 'tab_label': '在留外国人', 'col_head': '中国'}
{'col_cat': 'cat02', 'col_cat_label': '国籍・地域', 'restriction': None, 'tab_label': '在留外国人', 'col_head': '台湾'}
{'col_cat': 'cat02', 'col_cat_label': '国籍・地域', 'restriction': None, 'tab_label': '在留外国人', 'col_head': '韓国・朝鮮'}
{'col_cat': 'cat02', 'col_cat_label': '国籍・地域', 'restriction': None, 'tab_label': '在留外国人', 'col_head': 'フィリピン'}
{'col_cat': 'cat02', 'col_cat_label': '国籍・地域', 'restriction': None, 'tab_label': '在留外国人', 'col_head': 'ブラジル'}
{'col_cat': 'cat02', 'col_cat_label': '国籍・地域', 'rest

Now iterate through the rows and create column-oriented `pandas.DataFrame.

The check for the last row in the table looks for a missing cell in the leftmost column. The current example is missing one.  

In [14]:
import pandas as pd
import copy

n = col_headers[0].header_length()
saved_row_label = [None] * n  # a buffer for propagating abbreviated label members down the table
print(saved_row_label)

column_names = [
    '表章項目', # NOT title_result["tab_label"] ,
    title_result["cat01_label"],
    title_result["cat02_label"],
    'time_code',
    "value"
]
if title_result["sub_tab_label"] != None:
    column_names.append("表章小項目")

expected_cell_count = 0 # count the number of cells covered by the col_headers
for chead in col_headers:
    expected_cell_count = expected_cell_count + chead.header_length()

rows = []
bodyrowiter = ws.iter_rows(
    min_row=ws._print_rows.max_row+1, 
    max_row=ws.max_row
    )
br = next(bodyrowiter, None)
while br != None:
    # iterate through the col_headers, not the row
    hdridx = 0
    cellidx = 0
    rowhead = [
        title_result["tab_label"] # the value of the first column
    ]
    # the first col_header is the cat01 value
    hdr = col_headers[hdridx]
    hdcontent = hdr.serialize(ws)
    ridx = br[0].row
    cidx = hdr.cell_list[cellidx][1]
    cidxbnd = cidx + hdr.header_length()
    thislabel = ""
    cidxvar = cidx
    while cidxvar < cidxbnd:
        saved_row_label_idx = cidxvar-cidx
        c = ws.cell(ridx, cidxvar)
        if isinstance(c, Cell) == True:
            if c.value != None:
                saved_row_label[saved_row_label_idx] = c.value
            thislabel = thislabel + saved_row_label[saved_row_label_idx]
        else:
            break
        cidxvar = cidxvar + 1
    rowhead.append(thislabel)
    # the rest of the row are tabulated values
    hdridx = 1
    cellidx = cidxbnd
    while hdridx < len(col_headers) and cellidx < expected_cell_count:
        rowwhole = copy.deepcopy(rowhead)
        hdr = col_headers[hdridx]
        hdcontent = hdr.serialize(ws)
        thishead = hdcontent['col_head']
        rowwhole.append(thishead)
        # add time_code
        rowwhole.append(title_result["time_code"])
        ridx = br[0].row
        cidx = hdr.cell_list[0][1]
        cidxbnd = cidx + hdr.header_length()
        val = ""
        cidxvar = cidx
        while cidxvar < cidxbnd:
            c = ws.cell(ridx, cidxvar)
            if isinstance(c, Cell) == True:
                if c.value != None:
                    if isinstance(c.value, str):
                        val = val + str(c.value)
                    else:
                        val = c.value
            else:
                break
            cidxvar = cidxvar + 1
        rowwhole.append(val)
        # if there is a subcategory for the whole table, add it here
        if title_result["sub_tab_label"] != None:
            column_names.append(title_result["sub_tab_label"])
        # add to rows[]
        row_dict = {k: v for k, v in zip(column_names, rowwhole)}
        rows.append(row_dict)
        hdridx = hdridx + 1
        cellidx = cidxbnd
    br = next(bodyrowiter, None)
    if br != None:
        values = list(filter(lambda x: x.value != None, br ))
        if len(values) <= 1:
            br = None

df = pd.DataFrame.from_dict(rows)
df  


[None, None, None]


,表章項目,市区町村,国籍・地域,time_code,value
0,在留外国人,北海道札幌市,総数,2014001212,10024
1,在留外国人,北海道札幌市,中国,2014001212,3648
2,在留外国人,北海道札幌市,台湾,2014001212,250
3,在留外国人,北海道札幌市,韓国・朝鮮,2014001212,2572
4,在留外国人,北海道札幌市,フィリピン,2014001212,340
...,...,...,...,...,...
20650,在留外国人,不詳・不定,フィリピン,2014001212,52
20651,在留外国人,不詳・不定,ブラジル,2014001212,67
20652,在留外国人,不詳・不定,ベトナム,2014001212,43
20653,在留外国人,不詳・不定,ペルー,2014001212,15


In [15]:
local_var = '北海道札幌市'

df.dtypes

rows

[{'表章項目': '在留外国人',
  '市区町村': '北海道札幌市',
  '国籍・地域': '総数',
  'time_code': 2014001212,
  'value': 10024},
 {'表章項目': '在留外国人',
  '市区町村': '北海道札幌市',
  '国籍・地域': '中国',
  'time_code': 2014001212,
  'value': 3648},
 {'表章項目': '在留外国人',
  '市区町村': '北海道札幌市',
  '国籍・地域': '台湾',
  'time_code': 2014001212,
  'value': 250},
 {'表章項目': '在留外国人',
  '市区町村': '北海道札幌市',
  '国籍・地域': '韓国・朝鮮',
  'time_code': 2014001212,
  'value': 2572},
 {'表章項目': '在留外国人',
  '市区町村': '北海道札幌市',
  '国籍・地域': 'フィリピン',
  'time_code': 2014001212,
  'value': 340},
 {'表章項目': '在留外国人',
  '市区町村': '北海道札幌市',
  '国籍・地域': 'ブラジル',
  'time_code': 2014001212,
  'value': 60},
 {'表章項目': '在留外国人',
  '市区町村': '北海道札幌市',
  '国籍・地域': 'ベトナム',
  'time_code': 2014001212,
  'value': 138},
 {'表章項目': '在留外国人',
  '市区町村': '北海道札幌市',
  '国籍・地域': 'ペルー',
  'time_code': 2014001212,
  'value': 14},
 {'表章項目': '在留外国人',
  '市区町村': '北海道札幌市',
  '国籍・地域': '米国',
  'time_code': 2014001212,
  'value': 540},
 {'表章項目': '在留外国人',
  '市区町村': '北海道札幌市中央区',
  '国籍・地域': '総数',
  'time_code': 2014001212,
 